# TP 4 – Le fil rouge en DataFrames et Spark SQL (45 min)

**Objectif** : refaire le TP 2 avec l'API structurée, aller plus loin (heure, arrondissement), et comparer avec les RDD.

Données : `data/trips.parquet` et `data/taxi_zone_lookup.csv`.

**Réussi si** : le top 10 SQL est identique à celui du TP 2, et 31 dossiers `jour=…` sont écrits.

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("TP")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "—", spark.sparkContext.uiWebUrl)

### 1. Lire `trips.parquet` avec un **schéma explicite**

Gardez seulement `tpep_pickup_datetime`, `PULocationID`, `DOLocationID`, `trip_distance`, `total_amount`.

Indices : `StructType([StructField(nom, type), ...])`, `spark.read.schema(schema).parquet(...)`.
Les identifiants sont des entiers 32 bits (`IntegerType`), les montants des `DoubleType`,
l'horodatage est en heure locale sans fuseau (`TimestampNTZType`).

In [ ]:
from pyspark.sql.types import *
schema = StructType([
    ...
])
trips = ...
trips.printSchema()
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True)

### 2. Créer les vues `trips` et `zones` (`createOrReplaceTempView`)

In [ ]:
...

### 3a. En SQL : le top 10 des zones de départ par chiffre d'affaires, avec le nom de la zone et l'arrondissement

Indice : `JOIN zones z ON t.PULocationID = z.LocationID`, `GROUP BY`, `ORDER BY ... DESC LIMIT 10`.

In [ ]:
top10 = spark.sql("""
    SELECT ...
""")
top10.show(truncate=False)

### 3b. En SQL : nombre de courses et chiffre d'affaires par heure de la journée. À quelle heure New York rapporte-t-il le plus ?

Indice : `HOUR(tpep_pickup_datetime) AS heure`, `COUNT(*)`, `SUM(total_amount)`, `ORDER BY heure`.

In [ ]:
spark.sql("""
    SELECT ...
""").show(24)

### 4. Avec l'API DataFrame : classement des arrondissements (`Borough`) par chiffre d'affaires

Indice : `join`, `groupBy`, `agg(F.sum(...))`, `orderBy(F.desc(...))`.

In [ ]:
from pyspark.sql import functions as F
par_borough = ...
par_borough.show()

### 5. Écrire les courses en Parquet, **partitionnées par jour**, dans `out/trips_par_jour`

Indice : `withColumn("jour", F.to_date(...))`, `write.partitionBy("jour").parquet(...)`. Combien de dossiers obtenez-vous ?

In [ ]:
...

### Bonus 1 – `explain()`

Affichez le plan physique du top 10. Que fait Catalyst avec la jointure (indice : *Broadcast*) ? Comparez la lisibilité avec le TP 2, et le temps avec `import time ; t = time.time() ; top10.collect() ; print(time.time() - t)`.

In [ ]:
top10.explain()

### Bonus 2 – ANSI SQL dans Spark 4

Calculez `total_amount / trip_distance` **sur les courses de distance nulle** : `spark.sql("SELECT total_amount / trip_distance FROM trips WHERE trip_distance = 0").show()`.
Que se passe-t-il ? Corrigez avec `try_divide(total_amount, trip_distance)`.

Attention : sans le `WHERE`, un simple `show()` ne lit que les 20 premières lignes, qui n'ont pas de distance nulle : vous ne verriez aucune erreur. Spark est paresseux jusqu'au bout !

In [ ]:
...